**Lab type:** debug

**Course:** ML101 — Intro to Machine Learning

**Lesson:** Regression Analysis — Predicting Continuous Values

**Task:** The AI-generated analysis below contains 3 bugs. For each bug: identify what is wrong, explain why the output is misleading, and write the corrected code in the fix cell.

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score

np.random.seed(42)
n = 300
square_feet = np.random.normal(1500, 400, n).clip(500, 4000)
bedrooms = np.random.choice([1, 2, 3, 4, 5], n)
age = np.random.uniform(0, 50, n)
price = 150 * square_feet + 10000 * bedrooms - 500 * age + np.random.normal(0, 20000, n)
df = pd.DataFrame({'square_feet': square_feet, 'bedrooms': bedrooms, 'age': age, 'price': price})
print(f"Dataset shape: {df.shape}")
df.head()

## Step 1: Scaling and Splitting

Before training a linear regression model, we need to scale our features so that no single feature dominates due to its magnitude, then split the data into training and test sets.

In [ ]:
# AI-generated — scaling and splitting the dataset
scaler = StandardScaler()
df[['square_feet', 'bedrooms', 'age']] = scaler.fit_transform(df[['square_feet', 'bedrooms', 'age']])  # <- Bug 1
X = df[['square_feet', 'bedrooms', 'age']]
y = df['price']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Training samples: {len(X_train)}, Test samples: {len(X_test)}")
print("Features scaled and data split successfully.")

**Bug 1 Investigation:** Run the cell above. The code runs without errors and prints reassuring output — but something about the order of operations is wrong. Think carefully about what information the scaler has seen before the train/test split happens.

In [ ]:
# Fix Bug 1 here
# YOUR CODE HERE

**Explanation:** Write your answer here — what was wrong and why the fix is correct.

<details>
<summary>🔑 Reveal answer — Bug 1</summary>

**What was wrong:** `scaler.fit_transform` was called on the full dataset before `train_test_split`. This means the scaler computed its mean and standard deviation using test-set rows — the model has indirectly "seen" the test data during preprocessing (data leakage).

**Why it matters:** The scaled test features are no longer truly unseen; metrics computed on them are optimistically biased. In production the scaler would only have access to training statistics, so this pipeline does not reflect real performance.

**Correct approach:** Call `train_test_split` first, then `scaler.fit_transform(X_train)` and `scaler.transform(X_test)` — the scaler learns statistics only from training rows.

</details>

## Step 2: Model Evaluation

With the data scaled and split, we train a linear regression model and evaluate how well it captures the relationship between house features and price.

In [ ]:
# AI-generated — training the model and reporting performance
# Note: this cell assumes X_train / X_test are already correctly scaled and split
model = LinearRegression()
model.fit(X_train, y_train)
train_pred = model.predict(X_train)
r2 = r2_score(y_train, train_pred)  # <- Bug 2
print(f"Model R\u00b2: {r2:.3f}")
print("Model explains a high proportion of variance — looks great!")

**Bug 2 Investigation:** Run the cell above. The R² value looks impressive. But is this the number you should be reporting to a stakeholder? Consider what dataset was used to both train the model and compute this metric.

In [ ]:
# Fix Bug 2 here
# YOUR CODE HERE

**Explanation:** Write your answer here — what was wrong and why the fix is correct.

<details>
<summary>🔑 Reveal answer — Bug 2</summary>

**What was wrong:** R² was computed on `y_train` — the same data used to fit the model. A model can score a near-perfect R² on its own training set simply by memorising it, so this number is not a reliable indicator of generalisation.

**Why it matters:** Reporting train R² to a stakeholder overstates model quality. A model that overfits badly can still show R² ≈ 1.0 on training data.

**Correct approach:** Always report R² (and other metrics) on the held-out test set: `r2_score(y_test, model.predict(X_test))`. Comparing train R² to test R² together is even better — a large gap signals overfitting.

</details>

## Step 3: Comparing Models

Now we compare a simple single-feature model (square footage only) against the full multi-feature model to decide which to deploy.

In [ ]:
# AI-generated — comparing single-feature vs multi-feature model
# Note: this cell uses model trained in Step 2 and assumes X_train/X_test are available
model2 = LinearRegression()
model2.fit(X_train[['square_feet']], y_train)  # single feature
pred_simple = model2.predict(X_test[['square_feet']])
pred_multi = model.predict(X_test)
mse_simple = mean_squared_error(y_test, pred_simple)
mse_multi = mean_squared_error(y_test, pred_multi)
print(f"Simple model MSE:  {mse_simple:.0f}")
print(f"Multi-feature MSE: {mse_multi:.0f}")  # <- Bug 3
# The simple model MSE is larger, so the multi-feature model is better — comparison is clear.

**Bug 3 Investigation:** Run the cell above. The numbers printed are very large. What unit are they in? Can you intuitively tell a colleague how much better one model is than the other just from these numbers? Think about how MSE is computed and what a more interpretable alternative would be.

In [ ]:
# Fix Bug 3 here
# YOUR CODE HERE

**Explanation:** Write your answer here — what was wrong and why the fix is correct.

<details>
<summary>🔑 Reveal answer — Bug 3</summary>

**What was wrong:** The comparison used MSE (Mean Squared Error). MSE is in squared units — for a price model measured in dollars, MSE is in dollars², a number that has no intuitive meaning to a colleague.

**Why it matters:** Saying "the simple model's MSE is 4,000,000,000" tells a stakeholder nothing actionable. You cannot quickly judge whether that gap is large or small.

**Correct approach:** Use RMSE (`np.sqrt(MSE)`) which is in the same unit as the target (dollars). A statement like "the multi-feature model cuts the average prediction error by $3,400" is immediately interpretable.

</details>

## Corrected Analysis

The cell below applies all three fixes in the correct order. Run it end-to-end to confirm the pipeline is now sound.

In [ ]:
# --- Corrected pipeline: all three bugs fixed ---

# Fix 1: split BEFORE fitting the scaler
X_raw = df[['square_feet', 'bedrooms', 'age']]
y = df['price']
X_train_raw, X_test_raw, y_train, y_test = train_test_split(X_raw, y, test_size=0.2, random_state=42)

scaler_fixed = StandardScaler()
X_train = scaler_fixed.fit_transform(X_train_raw)  # fit only on training data
X_test = scaler_fixed.transform(X_test_raw)         # transform test with training statistics

# Fix 2: evaluate on BOTH train and test sets and compare
model_fixed = LinearRegression()
model_fixed.fit(X_train, y_train)

train_pred = model_fixed.predict(X_train)
test_pred = model_fixed.predict(X_test)

r2_train = r2_score(y_train, train_pred)
r2_test = r2_score(y_test, test_pred)
print(f"Train R\u00b2: {r2_train:.3f}  |  Test R\u00b2: {r2_test:.3f}")
print("  -> Similar values indicate the model generalises well (no overfitting).")

# Fix 3: report RMSE (same units as price) instead of MSE
model_simple = LinearRegression()
model_simple.fit(X_train[:, [0]], y_train)  # square_feet column only
pred_simple = model_simple.predict(X_test[:, [0]])

rmse_simple = np.sqrt(mean_squared_error(y_test, pred_simple))
rmse_multi = np.sqrt(mean_squared_error(y_test, test_pred))
print(f"\nSimple model RMSE:  ${rmse_simple:,.0f}")
print(f"Multi-feature RMSE: ${rmse_multi:,.0f}")
print(f"  -> The multi-feature model reduces average prediction error by ${rmse_simple - rmse_multi:,.0f}.")

<details>
<summary>🔑 Reveal summary answers</summary>

1. **Data leakage via early scaling:** Fit the scaler only on training data — scaling the full dataset before splitting lets test statistics influence training.

2. **Evaluate on the test set:** Train R² is always optimistic; report R² on the held-out test set so the metric reflects generalisation, not memorisation.

3. **Report RMSE, not MSE:** RMSE is in the same unit as the target, making model comparisons immediately interpretable to non-technical stakeholders.

</details>